# Maze schematics

A schematic of the connected-platform maze for one problem: initiation tower dark blue, choice towers light blue, the others white. A1 is the bottom-left tower, as in the maze heatmaps.

Choose the problem in one of two ways (set one in the cell below, leave the other `None`):

1. **From the data**: a cohort, mouse, task type and problem number. The problem number counts within the task type, as in the cohort figures ("drifting problem 6"), and the towers and sessions are read from the recordings in `<BANDIT_DATA_ROOT>/3x3_field_magnitude_bandit/rawdata/<cohort>`. Needs `BANDIT_DATA_ROOT` set (see `.env.example`); the first load of a cohort is slow, later ones come from the cache.
2. **By hand**: a list of choice towers, with the initiation tower at B2.

Figures are saved in `results/figures/maze-schematics/` (git-ignored). With `PAPER_READY = True` they use the lab's paper style, as `run_cohorts.py --paper-ready` does: Arial at 7 pt, a journal column width, editable text, PDF + SVG + PNG and a `.caption.txt` (the title moves into the caption); they go in a `paper-ready/` subfolder.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name in {"individual_analyses", "scripts"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from contextlib import nullcontext

from IPython.display import Image, display

from src.behavior_import.problem_numbering import DEFAULT_INITIATION_TOWER
from src.behavior_import.task_types import problem_layout, problem_map_table
from src.behavior_visualization.paper_style import paper_figures
from src.behavior_visualization.plot_maze_schematic import plot_maze_schematic
from src.pipelines.cohort_analysis import cohort_id_from_name, load_cohort

## Settings

`PAPER_READY`: draw in the paper style (see above) instead of the normal one.
`ADD_TITLE`: label the figure with the problem (and, from the data, its sessions); in paper style the title goes to the caption file.

In [ ]:
PAPER_READY = False
ADD_TITLE = False

OUTPUT_DIR = PROJECT_ROOT / "results" / "figures" / "maze-schematics"
if PAPER_READY:
    OUTPUT_DIR = OUTPUT_DIR / "paper-ready"

## Which problem

Set **one** of `COHORT` and `CHOICE_TOWERS`, and leave the other `None`.

In [ ]:
# 1. From the data: set COHORT (and the three below it).
COHORT = None  # e.g. "cohort-01"; None to give the towers by hand
SUBJECT = "MY_71_R"
TASK = "drifting"  # "drifting" or "blocked"
PROBLEM = 6  # counted within the task type, as in the cohort figures

# 2. By hand, with the initiation tower at B2 (the protocol default): set CHOICE_TOWERS.
CHOICE_TOWERS = ["A2", "B1", "C3"]  # None to read the problem from the data

if (COHORT is None) == (CHOICE_TOWERS is None):
    raise ValueError("set exactly one of COHORT and CHOICE_TOWERS")

## Towers, title and file name

In [ ]:
if COHORT is not None:
    imported = load_cohort(cohort_id_from_name(COHORT), data_root=None, use_cache=True)
    problem_map = problem_map_table(imported.trials_by_problem)
    layout = problem_layout(problem_map, mouse=SUBJECT, task_type=TASK, problem=PROBLEM)
    choice_towers, initiation_tower = layout.choice_towers, layout.initiation_tower
    title = f"{SUBJECT}: {TASK} problem {PROBLEM} (sessions {layout.session_range})"
    name = f"{COHORT}_{SUBJECT}_{TASK}-problem-{PROBLEM:02d}"
else:
    choice_towers, initiation_tower = CHOICE_TOWERS, DEFAULT_INITIATION_TOWER
    title = "Choice towers " + ", ".join(sorted(CHOICE_TOWERS))
    name = "towers-" + "-".join(sorted(CHOICE_TOWERS))

print(f"Initiation tower {initiation_tower}; choice towers {', '.join(choice_towers)}")

## Draw and save

In [ ]:
base = OUTPUT_DIR / name
with paper_figures() if PAPER_READY else nullcontext():
    plot_maze_schematic(
        choice_towers, initiation_tower, title=title if ADD_TITLE else None, save_path=base
    )
print("Saved", base.with_suffix(".svg"))
display(Image(filename=base.with_suffix(".png")))